# Moduł 14: Przetwarzanie audio

**Wymagane przez syllabus IOAI 2026** (kategoria Practice): Pre-trained Audio Encoders (HuBERT), Audio Models (Whisper, Qwen-Audio, Voxtral).

## Spis treści
1. Podstawy audio i reprezentacje
2. Spektrogramy i cechy mel
3. Whisper — rozpoznawanie mowy
4. HuBERT — self-supervised audio encoder
5. Qwen-Audio i Voxtral — multimodalne modele audio
6. Klasyfikacja audio
7. **Zadania**

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Audio-specific
try:
 import torchaudio
 print(f"torchaudio version: {torchaudio.__version__}")
except ImportError:
 print("pip install torchaudio")

try:
 import librosa
 print(f"librosa version: {librosa.__version__}")
except ImportError:
 print("pip install librosa")

---

## 1. Podstawy audio

### Sygnał dźwiękowy

Dźwięk to fala ciśnienia. Po digitalizacji mamy **dyskretny sygnał** $x[n]$:

- **Sampling rate** (częstotliwość próbkowania): $f_s$ [Hz] — ile próbek na sekundę (np. 16000 Hz, 44100 Hz)
- **Twierdzenie Nyquista**: aby odtworzyć częstotliwość $f$, potrzeba $f_s \geq 2f$
- **Bit depth**: precyzja każdej próbki (16-bit, 24-bit, 32-bit float)
- **Kanały**: mono (1), stereo (2)

Na olimpiadzie audio zazwyczaj ma $f_s = 16000$ Hz (wystarczające dla mowy).

### Wczytywanie audio

```python
# torchaudio
waveform, sample_rate = torchaudio.load("audio.wav") # → [channels, time]

# librosa
y, sr = librosa.load("audio.wav", sr=16000) # → numpy, mono
```

### Resampling

Modele (np. Whisper) wymagają konkretnego $f_s$:
```python
resampler = torchaudio.transforms.Resample(orig_freq=44100, new_freq=16000)
waveform_16k = resampler(waveform)
```

In [ ]:
# === Generowanie syntetycznego sygnału audio ===
# (na olimpiadzie wczytasz prawdziwy plik)

sr = 16000 # sampling rate
duration = 2.0 # sekundy
t = np.linspace(0, duration, int(sr * duration), endpoint=False)

# Sygnał: suma sinusoid o różnych częstotliwościach
freq1, freq2, freq3 = 440, 880, 1320 # A4, A5, E6
signal = 0.5 * np.sin(2 * np.pi * freq1 * t) + \
 0.3 * np.sin(2 * np.pi * freq2 * t) + \
 0.2 * np.sin(2 * np.pi * freq3 * t)

# Dodaj szum
signal += 0.05 * np.random.randn(len(signal))

plt.figure(figsize=(12, 3))
plt.plot(t[:1600], signal[:1600]) # pierwsze 0.1s
plt.xlabel('Czas [s]')
plt.ylabel('Amplituda')
plt.title(f'Sygnał audio (pierwsze 0.1s) | sr={sr} Hz')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Kształt sygnału: {signal.shape}")
print(f"Długość: {len(signal)/sr:.1f}s")
print(f"Min/Max: {signal.min():.3f} / {signal.max():.3f}")

---

## 2. Spektrogramy i cechy mel

### Transformata Fouriera (STFT)

**Short-Time Fourier Transform** dzieli sygnał na krótkie okna i dla każdego oblicza FFT:

$$\text{STFT}(m, k) = \sum_{n=0}^{N-1} x[n + mH] \, w[n] \, e^{-j 2\pi kn/N}$$

- $m$ — indeks okna (frame)
- $k$ — indeks częstotliwości (bin)
- $N$ — rozmiar FFT (np. 512, 1024, 2048)
- $H$ — hop length (przesunięcie okna, np. 512)
- $w[n]$ — funkcja okna (Hann, Hamming)

**Spektrogram** = $|\text{STFT}|^2$ — moc w funkcji czasu i częstotliwości.

### Skala mel

Ludzkie ucho postrzega częstotliwości **logarytmicznie**. Skala mel:

$$m = 2595 \log_{10}\left(1 + \frac{f}{700}\right)$$

**Mel-spectrogram**: spektrogram przefiltrowany przez **mel filter bank** — zestaw trójkątnych filtrów rozmieszczonych w skali mel.

### Log-mel spectrogram

$$\text{log-mel} = \log(\text{mel-spectrogram} + \epsilon)$$

Jest to **standardowe wejście** do modeli audio (Whisper, HuBERT, itp.).

### MFCC (Mel-Frequency Cepstral Coefficients)

Dalsza kompresja: DCT na log-mel → 13-40 współczynników na frame.
Rzadziej używane w deep learning (mel-spectrogram jest preferowany).

In [ ]:
# === Obliczanie i wizualizacja spektrogramów ===

signal_tensor = torch.tensor(signal, dtype=torch.float32).unsqueeze(0) # [1, T]

# 1. Spectrogram (STFT)
n_fft = 1024
hop_length = 256

try:
 spectrogram_transform = torchaudio.transforms.Spectrogram(
 n_fft=n_fft, hop_length=hop_length, power=2.0
 )
 spec = spectrogram_transform(signal_tensor)

 # 2. Mel-spectrogram
 mel_transform = torchaudio.transforms.MelSpectrogram(
 sample_rate=sr, n_fft=n_fft, hop_length=hop_length, n_mels=80
 )
 mel_spec = mel_transform(signal_tensor)

 # 3. Log-mel spectrogram
 log_mel = torch.log(mel_spec + 1e-9)

 fig, axes = plt.subplots(1, 3, figsize=(18, 4))

 axes[0].imshow(10 * torch.log10(spec[0] + 1e-9).numpy(), aspect='auto', origin='lower', cmap='viridis')
 axes[0].set_title('Spectrogram (dB)')
 axes[0].set_xlabel('Czas (frames)')
 axes[0].set_ylabel('Częstotliwość (bins)')

 axes[1].imshow(mel_spec[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
 axes[1].set_title('Mel-Spectrogram')
 axes[1].set_xlabel('Czas (frames)')
 axes[1].set_ylabel('Mel bins')

 axes[2].imshow(log_mel[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
 axes[2].set_title('Log-Mel Spectrogram')
 axes[2].set_xlabel('Czas (frames)')
 axes[2].set_ylabel('Mel bins')

 plt.suptitle('Reprezentacje audio')
 plt.tight_layout()
 plt.show()

 print(f"Spectrogram shape: {spec.shape}")
 print(f"Mel-spectrogram shape: {mel_spec.shape}")
 print(f"Log-mel shape: {log_mel.shape}")

except NameError:
 print("torchaudio not available, using manual computation")
 # FFT-based spectrogram
 from scipy.signal import stft
 f, t_stft, Zxx = stft(signal, fs=sr, nperseg=n_fft, noverlap=n_fft-hop_length)
 plt.figure(figsize=(10, 4))
 plt.pcolormesh(t_stft, f, np.abs(Zxx), shading='gouraud')
 plt.ylabel('Freq [Hz]')
 plt.xlabel('Time [s]')
 plt.title('Spectrogram (scipy)')
 plt.colorbar()
 plt.show()

---

## 3. Whisper — rozpoznawanie mowy

**Whisper** (OpenAI, 2022) to encoder-decoder Transformer do:
- Automatic Speech Recognition (ASR) — transkrypcja mowy
- Speech Translation — tłumaczenie mowy
- Language Detection
- Voice Activity Detection

### Architektura

1. **Wejście**: Log-mel spectrogram (80 mel bins, 30s okna)
2. **Audio Encoder**: Transformer encoder (Conv1d stem → Transformer blocks)
3. **Text Decoder**: Autoregressive Transformer decoder
4. **Wyjście**: tokeny tekstowe (BPE)

### Warianty

| Model | Parametry | Relative Speed | English WER |
|-------|-----------|---------------|-------------|
| tiny | 39M | 32x | ~8% |
| base | 74M | 16x | ~5.6% |
| small | 244M | 6x | ~4.3% |
| medium | 769M | 2x | ~3.5% |
| large-v3 | 1550M | 1x | ~2.7% |
| turbo | 809M | 8x | ~3% |

### Użycie

```python
import whisper

model = whisper.load_model("base")
result = model.transcribe("audio.mp3")
print(result["text"])
```

Lub z Hugging Face `transformers`:
```python
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained("openai/whisper-base")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base")

input_features = processor(audio_array, sampling_rate=16000, return_tensors="pt").input_features
predicted_ids = model.generate(input_features)
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
```

### Tokeny specjalne Whisper

```
<|startoftranscript|> <|en|> <|transcribe|> <|notimestamps|> text... <|endoftext|>
```

- `<|en|>` → język
- `<|transcribe|>` vs `<|translate|>` → tryb
- `<|0.00|>` → timestampy

In [ ]:
# === Whisper — demonstracja ===

try:
 from transformers import WhisperProcessor, WhisperForConditionalGeneration

 # Użyj least model (tiny/base) na olimpiadzie jeśli GPU słabe
 model_name = "openai/whisper-tiny"
 processor = WhisperProcessor.from_pretrained(model_name)
 model = WhisperForConditionalGeneration.from_pretrained(model_name)
 model.eval()

 # Synthetyczny sygnał (na olimpiadzie: wczytaj prawdziwy plik)
 # signal to nasz syntetyczny sygnał z sekcji 1
 input_features = processor(
 signal.astype(np.float32), 
 sampling_rate=16000, 
 return_tensors="pt"
 ).input_features

 print(f"Input features shape: {input_features.shape}")
 print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

 # Generuj transkrypcję
 with torch.no_grad():
 predicted_ids = model.generate(input_features)

 transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
 print(f"\nTranskrypcja: {transcription[0]}")
 print("(syntetyczny sygnał → prawdopodobnie nonsens, ale model działa!)")

 # Encoder embeddings (jako feature extractor)
 with torch.no_grad():
 encoder_outputs = model.model.encoder(input_features)
 audio_embedding = encoder_outputs.last_hidden_state.mean(dim=1) # [1, d_model]

 print(f"\nAudio embedding shape: {audio_embedding.shape}")
 print("→ Można użyć jako feature do klasyfikacji audio")

except ImportError:
 print("transformers nie jest zainstalowany.")
 print("pip install transformers")
 print("\n Na olimpiadzie: Whisper jest zwykle dostępny offline.")

---

## 4. HuBERT — Self-Supervised Audio Encoder

**HuBERT** (Hidden-Unit BERT, Meta 2021) to self-supervised model audio, analogiczny do BERT dla tekstu.

### Idea

1. **Pseudo-labeling**: K-Means na cechach MFCC → dyskretne "tokeny" audio
2. **Masked prediction**: maskuj fragmenty waveform, predykuj pseudo-tokeny
3. **Iteracja**: retrain K-Means na lepszych cechach → lepsze labele → lepszy model

### Architektura

- **CNN feature extractor**: 7 warstw Conv1d na raw waveform → 20ms frames
- **Transformer encoder**: 12/24 warstw self-attention
- **Wyjście**: reprezentacje audio na poziomie frame (20ms)

### Zastosowania HuBERT

| Zadanie | Opis | Podejście |
|---------|------|----------|
| Speech Recognition (ASR) | Transkrypcja mowy | Fine-tune z CTC loss |
| Speaker Identification | Kto mówi? | Embeddings → classifier |
| Emotion Recognition | Emocje w głosie | Embeddings → classifier |
| Audio Classification | Typ dźwięku | Embeddings → classifier |

### Użycie jako feature extractor

```python
from transformers import HubertModel, Wav2Vec2FeatureExtractor

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/hubert-base-ls960")
model = HubertModel.from_pretrained("facebook/hubert-base-ls960")

inputs = feature_extractor(audio_array, sampling_rate=16000, return_tensors="pt")
with torch.no_grad():
 outputs = model(**inputs)
 embeddings = outputs.last_hidden_state # [batch, time_frames, 768]
 # Agregacja: mean pooling
 audio_embedding = embeddings.mean(dim=1) # [batch, 768]
```

In [ ]:
# === HuBERT — użycie jako feature extractor ===

try:
 from transformers import HubertModel, Wav2Vec2FeatureExtractor

 model_name = "facebook/hubert-base-ls960"
 feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
 hubert_model = HubertModel.from_pretrained(model_name)
 hubert_model.eval()

 # Przetwarzanie
 inputs = feature_extractor(
 signal.astype(np.float32),
 sampling_rate=16000,
 return_tensors="pt",
 padding=True
 )

 with torch.no_grad():
 outputs = hubert_model(**inputs)

 embeddings = outputs.last_hidden_state # [1, T_frames, 768]
 pooled = embeddings.mean(dim=1) # [1, 768]

 print(f"HuBERT output shape: {embeddings.shape}")
 print(f"Pooled embedding: {pooled.shape}")
 print(f"Model params: {sum(p.numel() for p in hubert_model.parameters()):,}")

 # Wizualizacja feature map
 plt.figure(figsize=(12, 4))
 plt.imshow(embeddings[0, :100, :100].numpy(), aspect='auto', cmap='viridis')
 plt.xlabel('Feature dim')
 plt.ylabel('Time frame')
 plt.title('HuBERT embeddings (pierwsze 100 frames × 100 dims)')
 plt.colorbar()
 plt.show()

except ImportError:
 print("transformers nie jest zainstalowany.")
 print("pip install transformers")

---

## 5. Qwen-Audio i Voxtral — multimodalne modele audio

### Qwen-Audio (Alibaba, 2023)

Model multimodalny rozumiejący audio + tekst:
- Architektura: Audio Encoder (Whisper-large encoder) + LLM (Qwen)
- Obsługuje: ASR, Audio QA, Sound Understanding, Music Understanding
- Może odpowiadać na pytania o treść audio

### Qwen2-Audio

Ulepszona wersja z dwoma trybami:
1. **Voice chat**: interakcja głosowa
2. **Audio analysis**: analiza audio z instrukcją tekstową

```python
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor

processor = AutoProcessor.from_pretrained("Qwen/Qwen2-Audio-7B-Instruct")
model = Qwen2AudioForConditionalGeneration.from_pretrained(
 "Qwen/Qwen2-Audio-7B-Instruct", torch_dtype=torch.float16, device_map="auto"
)

# Zapytanie o audio
messages = [
 {"role": "user", "content": [
 {"type": "audio", "audio_url": "path/to/audio.wav"},
 {"type": "text", "text": "What is being said in this audio?"}
 ]}
]
```

### Voxtral (Mistral, 2024)

Model od Mistral AI do rozumienia mowy:
- Bazuje na architekturze Mistral
- Audio encoder + LLM decoder
- Obsługuje: transkrypcja, tłumaczenie, analiza mowy

### Porównanie modeli audio

| Model | Typ | Wejście | Wyjście | Zastosowanie |
|-------|-----|---------|---------|-------------|
| **Whisper** | Encoder-Decoder | Audio → mel | Tekst | ASR, translation |
| **HuBERT** | Encoder only | Raw waveform | Embeddings | Feature extraction, fine-tune |
| **Qwen2-Audio** | Audio + LLM | Audio + tekst | Tekst | Audio QA, understanding |
| **Voxtral** | Audio + LLM | Audio + tekst | Tekst | Speech understanding |

### Kiedy którego użyć?

- **Transkrypcja** → Whisper (najlepszy ASR)
- **Feature extraction** → HuBERT (self-supervised embeddings)
- **Audio Q&A** → Qwen2-Audio (multimodalny)
- **Klasyfikacja dźwięków** → HuBERT embeddings + classifier

---

## 6. Klasyfikacja audio

### Wzorzec: pretrained encoder + classifier

Najbardziej typowe podejście na olimpiadzie:

1. **Załaduj pretrained audio encoder** (HuBERT lub Whisper encoder)
2. **Wyciągnij embeddingi** dla każdego pliku audio
3. **Wytrenuj klasyfikator** (Linear, MLP, lub nawet Random Forest)

### Podejście mel-spectrogram + CNN

Alternatywnie, traktuj audio jako obraz:
1. Oblicz **log-mel spectrogram** → tensor 2D
2. Użyj **pretrained CNN** (ResNet-18) do klasyfikacji
3. Fine-tune lub feature extraction

### Audio augmentacje

| Augmentacja | Opis |
|------------|------|
| Time stretch | Przyspieszenie/spowolnienie bez zmiany pitch |
| Pitch shift | Zmiana wysokości tonu |
| Add noise | Dodanie szumu Gaussowskiego |
| Time masking | Zamaskowanie losowego fragmentu czasu |
| Frequency masking | Zamaskowanie losowego pasma częstotliwości |
| SpecAugment | Time masking + Frequency masking (Google, 2019) |

In [ ]:
# === Klasyfikacja audio: mel-spectrogram + CNN ===
# Wzorzec olimpiadowy: traktuj audio jako obraz!

class AudioClassifierCNN(nn.Module):
 """Prosty CNN na mel-spektrogramie."""
 def __init__(self, n_mels=80, num_classes=10):
 super().__init__()
 self.features = nn.Sequential(
 nn.Conv2d(1, 32, kernel_size=3, padding=1),
 nn.ReLU(),
 nn.MaxPool2d(2),
 nn.Conv2d(32, 64, kernel_size=3, padding=1),
 nn.ReLU(),
 nn.MaxPool2d(2),
 nn.Conv2d(64, 128, kernel_size=3, padding=1),
 nn.ReLU(),
 nn.AdaptiveAvgPool2d((4, 4)), # stały rozmiar wyjściowy
 )
 self.classifier = nn.Sequential(
 nn.Flatten(),
 nn.Linear(128 * 4 * 4, 256),
 nn.ReLU(),
 nn.Dropout(0.3),
 nn.Linear(256, num_classes)
 )

 def forward(self, x):
 # x: [batch, 1, n_mels, time_frames]
 x = self.features(x)
 x = self.classifier(x)
 return x

model = AudioClassifierCNN(n_mels=80, num_classes=10)
print(f"Audio CNN params: {sum(p.numel() for p in model.parameters()):,}")

# Test z losowym wejściem
dummy_mel = torch.randn(4, 1, 80, 128) # batch=4, 1 kanał, 80 mels, 128 frames
out = model(dummy_mel)
print(f"Input shape: {dummy_mel.shape}")
print(f"Output shape: {out.shape}")

In [ ]:
# === Klasyfikacja audio: HuBERT embeddings + Linear ===
# Wzorzec olimpiadowy: feature extraction z pretrained model

class HuBERTClassifier(nn.Module):
 """HuBERT encoder (frozen) + linear classifier."""
 def __init__(self, num_classes=10, freeze_encoder=True):
 super().__init__()
 try:
 from transformers import HubertModel
 self.encoder = HubertModel.from_pretrained("facebook/hubert-base-ls960")
 if freeze_encoder:
 for param in self.encoder.parameters():
 param.requires_grad = False
 self.classifier = nn.Linear(768, num_classes)
 self.available = True
 except ImportError:
 self.available = False
 print("transformers not available")

 def forward(self, input_values):
 if not self.available:
 return None
 outputs = self.encoder(input_values)
 # Mean pooling over time
 embeddings = outputs.last_hidden_state.mean(dim=1) # [batch, 768]
 logits = self.classifier(embeddings) # [batch, num_classes]
 return logits

# Wzorzec użycia:
print("Wzorzec HuBERT + Classifier:")
print("1. feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(...)")
print("2. inputs = feature_extractor(audio, sr=16000, return_tensors='pt')")
print("3. logits = model(inputs.input_values)")
print("4. pred = logits.argmax(dim=-1)")

In [ ]:
# === SpecAugment — augmentacja spektrogramów ===

def spec_augment(spec, num_time_masks=2, num_freq_masks=2, 
 time_mask_param=20, freq_mask_param=10):
 """
 SpecAugment: maskowanie fragmentów spektrogramu.
 spec: [n_mels, time_frames]
 """
 spec = spec.clone()
 n_mels, n_frames = spec.shape

 # Time masking
 for _ in range(num_time_masks):
 t = np.random.randint(0, min(time_mask_param, n_frames))
 t0 = np.random.randint(0, max(1, n_frames - t))
 spec[:, t0:t0+t] = 0

 # Frequency masking
 for _ in range(num_freq_masks):
 f = np.random.randint(0, min(freq_mask_param, n_mels))
 f0 = np.random.randint(0, max(1, n_mels - f))
 spec[f0:f0+f, :] = 0

 return spec

# Wizualizacja SpecAugment
try:
 mel_transform = torchaudio.transforms.MelSpectrogram(
 sample_rate=sr, n_fft=1024, hop_length=256, n_mels=80
 )
 mel = mel_transform(signal_tensor)[0] # [80, T]
 log_mel = torch.log(mel + 1e-9)

 fig, axes = plt.subplots(1, 3, figsize=(15, 4))
 axes[0].imshow(log_mel.numpy(), aspect='auto', origin='lower')
 axes[0].set_title('Oryginał')

 for i in range(1, 3):
 augmented = spec_augment(log_mel, num_time_masks=2, num_freq_masks=2,
 time_mask_param=30, freq_mask_param=15)
 axes[i].imshow(augmented.numpy(), aspect='auto', origin='lower')
 axes[i].set_title(f'SpecAugment {i}')

 for ax in axes:
 ax.set_xlabel('Time')
 ax.set_ylabel('Mel bins')

 plt.suptitle('SpecAugment: maskowanie czasu i częstotliwości')
 plt.tight_layout()
 plt.show()

except NameError:
 print("torchaudio needed for SpecAugment visualization")

---

## Zadania do samodzielnego rozwiązania

---

### Zadanie 1: Spektrogram od zera (średnie)

Zaimplementuj własny mel-spectrogram **bez** torchaudio/librosa:
1. Oblicz STFT z oknem Hanna: $w[n] = 0.5(1 - \cos(2\pi n / (N-1)))$
2. Stwórz mel filter bank (trójkątne filtry)
3. Przelicz → mel-spectrogram → log-mel
4. Porównaj wizualnie z `torchaudio.transforms.MelSpectrogram`

In [ ]:
# Zadanie 1: Mel-spectrogram od zera
# TWÓJ KOD TUTAJ

### Zadanie 2: Klasyfikacja dźwięków (średnie)

Użyj zbioru [ESC-50](https://github.com/karolpiczak/ESC-50) lub syntetycznego:
1. Wygeneruj 3 klasy syntetycznych dźwięków:
 - Klasa 0: sinusoida 440 Hz + szum
 - Klasa 1: chirp (częstotliwość rośnie liniowo 200→2000 Hz)
 - Klasa 2: szum biały
2. Oblicz log-mel spectrogramy
3. Trenuj `AudioClassifierCNN` z tego modułu
4. Raportuj accuracy

In [ ]:
# Zadanie 2: Klasyfikacja dźwięków
# TWÓJ KOD TUTAJ

### Zadanie 3: Whisper transkrypcja (łatwe, wymaga transformers)

1. Załaduj Whisper tiny/base
2. Wczytaj plik audio (lub nagraj przez mikrofon/tts)
3. Transkrybuj z różnymi parametrami:
 - Wymuś język polski: `forced_decoder_ids = processor.get_decoder_prompt_ids(language="pl", task="transcribe")`
 - Tryb translate (tłumaczenie → angielski)
4. Porównaj wyniki

In [ ]:
# Zadanie 3: Whisper transcription
# TWÓJ KOD TUTAJ

### Zadanie 4: HuBERT embeddings + klasyfikator (trudne)

1. Wygeneruj syntetyczne dane (jak w Zadaniu 2)
2. Wyciągnij embeddingi z HuBERT (mean pooled)
3. Wytrenuj:
 a) Linear classifier (nn.Linear)
 b) Random Forest (sklearn)
4. Porównaj accuracy obu podejść z CNN na mel-spektrogramie

In [ ]:
# Zadanie 4: HuBERT embeddings
# TWÓJ KOD TUTAJ

### Zadanie 5: Augmentacja audio (srednie)

Zaimplementuj 4 techniki augmentacji audio:
1. **Time stretching** — zmiana tempa bez zmiany pitch
2. **Pitch shifting** — zmiana wysokosci dzwieku
3. **Dodanie szumu** — bialy szum o kontrolowanym SNR
4. **Time masking** — wyzerowanie losowego fragmentu (jak SpecAugment)

Dla kazdej techniki narysuj spektrogram oryginalnego i zaugmentowanego sygnalu.

In [ ]:
# Zadanie 5: Augmentacja audio
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

# Generuj prosty sygnal (suma sinusoid)
sr = 16000
t = np.linspace(0, 2, 2 * sr)
signal = np.sin(2 * np.pi * 440 * t) + 0.5 * np.sin(2 * np.pi * 880 * t)

# TWOJ KOD TUTAJ

### Zadanie 6: Ekstrakcja MFCC i klasyfikacja
Wygeneruj syntetyczne sygnały audio (sinus, piła, szum biały, chirp) po 1 sekundzie.
1. Oblicz MFCC (13 współczynników) dla każdego sygnału
2. Zwizualizuj MFCC jako heatmapy
3. Wytrenuj prosty klasyfikator (MLP) na cechach MFCC
4. Jaka jest accuracy na zbiorze testowym?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# TWOJ KOD TUTAJ


### Zadanie 7: Porównanie reprezentacji audio
Dla tego samego sygnału audio (np. sinus 440 Hz z szumem):
1. Oblicz i wyświetl: waveform, spektrogram, mel-spektrogram, MFCC
2. Zbadaj wpływ parametrów: window size (256, 512, 1024, 2048) na rozdzielczość
3. Pokaż trade-off rozdzielczość czasowa vs częstotliwościowa

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# TWOJ KOD TUTAJ


---

## Dodatek OAI: Ćwiczenia w stylu olimpiadowym

### Kontekst olimpiadowy
Audio w IOAI to stosunkowo nowy typ zadania. Kluczowe wzorce:
- Dane: pliki .wav/.mp3 z etykietami (klasyfikacja) lub transkrypcjami (ASR)
- Pipeline: wczytaj audio → preprocessing → model → predykcja
- Pretrained models oszczędzają czas i dają lepsze wyniki

### Ćwiczenie OAI-14A: Audio Classifier Pipeline

Napisz kompletny pipeline klasyfikacji audio:
```
load_audio() → resample_to_16k() → compute_mel() → augment() → model() → predict()
```
Obsłuż: zmienną długość plików (padding/truncation), batch processing.

### Ćwiczenie OAI-14B: Whisper Fine-tune (trudne)

Fine-tune Whisper tiny na prostym zbiorze:
1. Wygeneruj 100 próbek (pary: audio + tekst)
2. Zamroź encoder, trenuj decoder
3. Użyj `Seq2SeqTrainer` z Hugging Face

### Ćwiczenie OAI-14C: Multimodal Audio+Text

Zbuduj model łączący audio i tekst:
1. Audio embedding (HuBERT mean pool) → [768]
2. Text embedding (BERT [CLS]) → [768]
3. Concat → MLP → klasyfikacja
4. Porównaj: audio only vs text only vs multimodal

In [ ]:
# Ćwiczenie OAI-17A/B/C: Twój kod
# TWÓJ KOD TUTAJ